In [1]:
!pip install ccxt boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 8.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 11.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [boto3]m13/15 [ccxt]tp]]


In [ ]:
import ccxt
import pandas as pd
import time
import io
import boto3
from botocore.client import Config

MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "password123"
BUCKET_NAME = "crypto-raw-data"

s3_client = boto3.client(
    's3',
    endpoint_url="http://minio:9000", 
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4')
)

try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"ℹ️ Bucket '{BUCKET_NAME}' đã tồn tại.")
except:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Đã tự động tạo mới Bucket: '{BUCKET_NAME}' trên MinIO.")
exchange = ccxt.bitstamp()

try:
    btc_ohlcv = exchange.fetch_ohlcv('BTC/USD', '1m', limit=1000)
    df_btc = pd.DataFrame(btc_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df_btc['timestamp'] = pd.to_datetime(df_btc['timestamp'], unit='ms')

    csv_buffer = io.StringIO()
    df_btc.to_csv(csv_buffer, index=False)
    
    s3_client.put_object(Bucket=BUCKET_NAME, Key='bitcoin_1m.csv', Body=csv_buffer.getvalue())
    print(f"🚀 Đạt yêu cầu: Đã đẩy thành công 'bitcoin_1m.csv' ({len(df_btc)} dòng) lên MinIO!")
except Exception as e:
    print(f"❌ Lỗi xử lý BTC: {e}")
altcoins = [
    'ETH/USD', 'XRP/USD', 'LTC/USD', 'LINK/USD', 'UNI/USD',
    'MATIC/USD', 'SOL/USD', 'ADA/USD', 'DOT/USD', 'AVAX/USD',
    'DOGE/USD', 'SHIB/USD', 'BCH/USD', 'ALGO/USD', 'AAVE/USD'
]
df_alts_list = []

for coin in altcoins:
    try:
        ohlcv = exchange.fetch_ohlcv(coin, '1d', limit=500)
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df['symbol'] = coin 
        
        df_alts_list.append(df)
        print(f"  -> Đã tải xong {coin}")
        time.sleep(1) 
    except Exception as e:
        print(f"  ❌ Lỗi khi tải {coin}: {e}")

if df_alts_list:
    df_all_alts = pd.concat(df_alts_list, ignore_index=True)
    df_all_alts = df_all_alts[['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume']]
    

    alt_buffer = io.StringIO()
    df_all_alts.to_csv(alt_buffer, index=False)
    s3_client.put_object(Bucket=BUCKET_NAME, Key='altcoins_500d.csv', Body=alt_buffer.getvalue())


========== PIPELINE INGESTION: CRAWL & PUSH TO MINIO ==========
✅ Đã tự động tạo mới Bucket: 'crypto-raw-data' trên MinIO.

[1/2] Đang xử lý dữ liệu BTC/USD (1 phút)...
🚀 Đạt yêu cầu: Đã đẩy thành công 'bitcoin_1m.csv' (1000 dòng) lên MinIO!

[2/2] Đang xử lý dữ liệu 15 Altcoin (1 ngày)...
  -> Đã tải xong ETH/USD
  -> Đã tải xong XRP/USD
  -> Đã tải xong LTC/USD
  -> Đã tải xong LINK/USD
  -> Đã tải xong UNI/USD
  ❌ Lỗi khi tải MATIC/USD: bitstamp does not have market symbol MATIC/USD
  -> Đã tải xong SOL/USD
  -> Đã tải xong ADA/USD
  -> Đã tải xong DOT/USD
  -> Đã tải xong AVAX/USD
  -> Đã tải xong DOGE/USD
  -> Đã tải xong SHIB/USD
  -> Đã tải xong BCH/USD
  -> Đã tải xong ALGO/USD
  -> Đã tải xong AAVE/USD

🚀 Đạt yêu cầu: Đã đẩy thành công 'altcoins_500d.csv' (7000 dòng) lên MinIO!

==================== PIPELINE HOÀN THÀNH MỸ MÃN ====================
